In [32]:
# Import libraries

from pypdf import PdfReader
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import re
from collections import Counter

In [33]:
# Extract text from both PDF articles

def extract_text(filename):
    reader = PdfReader(filename)
    text = ""

    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"

    return text

australia_text = extract_text("Australia.pdf")
global_text = extract_text("Global Climate.pdf")


In [34]:
# Check word counts and preview extracted text

print("Australia word count:", len(australia_text.split()))
print("Global Climate word count:", len(global_text.split()))

print("\nAustralia text preview:")
print(australia_text[:500])

print("\nGlobal Climate text preview:")
print(global_text[:500])

Australia word count: 4337
Global Climate word count: 3564

Australia text preview:
COUNTRY/REGION REPORTS
Australia
(1) INTRODUCTION
An algae bloom in the continent’s southern waters and coral bleaching in the north formed 
the economic and environmental backdrop to Australia’s implementation of international 
environmental law this year. The May federal election returned the Labor government, 
which passed long-awaited reforms to the Environment Protection and Biodiversity 
Conservation Act 1999. These reforms, which have been discussed in these reports over 
many years, incl

Global Climate text preview:
AIR AND ATMOSPHERE 
Global Climate
(1) INTRODUCTION
Although climate change is perhaps the most widespread and global of all environmental 
challenges, states have not yet succeeded in implementing measures capable of delivering 
substantial, quantifiable results. Once again, numerous historical records across key indica-
tors of global climate deterioration were broken this year. I

In [35]:
# Clean and preprocess text

from nltk.corpus import stopwords

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    words = text.split()

    words = [word for word in words
             if word not in stop_words and len(word) > 2]

    return words

australia_words = clean_text(australia_text)
global_words = clean_text(global_text)

print("Australia:", australia_words[:30])
print("Global Climate:", global_words[:30])

Australia: ['country', 'region', 'reports', 'australia', 'introduction', 'algae', 'bloom', 'continent', 'southern', 'waters', 'coral', 'bleaching', 'north', 'formed', 'economic', 'environmental', 'backdrop', 'australia', 'implementation', 'international', 'environmental', 'law', 'year', 'may', 'federal', 'election', 'returned', 'labor', 'government', 'passed']
Global Climate: ['air', 'atmosphere', 'global', 'climate', 'introduction', 'although', 'climate', 'change', 'perhaps', 'widespread', 'global', 'environmental', 'challenges', 'states', 'yet', 'succeeded', 'implementing', 'measures', 'capable', 'delivering', 'substantial', 'quantifiable', 'results', 'numerous', 'historical', 'records', 'across', 'key', 'indicators', 'global']


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [36]:
# Compare legal and regulatory terminology

legal_keywords = [
    "law", "laws", "legislation", "regulation",
    "regulations", "act", "policy", "treaty",
    "agreement", "obligations", "compliance",
    "enforcement", "court", "rights",
    "emissions", "protection", "international",
    "national", "government", "targets"
]

comparison = pd.DataFrame({
    "Keyword": legal_keywords,
    "Australia": [australia_freq[word]
                  for word in legal_keywords],
    "Global Climate": [global_freq[word]
                       for word in legal_keywords]
})

comparison

,Keyword,Australia,Global Climate
0,law,20,19
1,laws,5,4
2,legislation,2,1
3,regulation,1,0
4,regulations,1,0
5,act,17,1
6,policy,2,5
7,treaty,4,2
8,agreement,9,11
9,obligations,6,18


In [38]:
# Keyword in context analysis

def keyword_context(text, keyword, window=150):
    text = re.sub(r"\s+", " ", text)

    for match in re.finditer(
        r"\b" + re.escape(keyword) + r"\b",
        text,
        re.IGNORECASE
    ):
        start = max(0, match.start() - window)
        end = min(len(text), match.end() + window)

        print(text[start:end])
        print("-" * 60)

print("AUSTRALIA:")
keyword_context(australia_text, "obligations")

print("\nGLOBAL CLIMATE:")
keyword_context(global_text, "obligations")

AUSTRALIA:
se gas emissions for major projects. Such a mechanism seems increasingly necessary after the International Court of Justice’s advisory opinion on the obligations of states with respect to climate change. Though non-binding, this 2025 opinion affirmed that states must protect the climate system through limiting
------------------------------------------------------------
e in the next five years to meet the 2030 target and triple in the decade leading to 2035 to reach the newly set goals. It also highlighted the legal obligations of states to protect the climate system as noted by the International Court of Justice (ICJ) in its advisory opinion. The federal government launched
------------------------------------------------------------
.com/yielaw/article/36/1/yvag046/8817270 by guest on 19 November 2026 Potentially diminishing Australia’s implementation of its international climate obligations was a legal development within the Australian states and territories this year. T

In [40]:
# Identify shared legal agreements and institutions

legal_phrases = [
    "Paris Agreement",
    "International Court of Justice",
    "ICJ",
    "human rights",
    "customary international law",
    "advisory opinion",
    "climate change",
    "legal obligations",
    "international law",
    "Environment Protection and Biodiversity Conservation Act"
]

def count_phrase(text, phrase):
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"\s+", " ", text)

    return len(re.findall(
        r"\b" + re.escape(phrase) + r"\b",
        text,
        re.IGNORECASE
    ))

legal_comparison = pd.DataFrame({
    "Legal Term": legal_phrases,
    "Australia": [count_phrase(australia_text, p)
                  for p in legal_phrases],
    "Global Climate": [count_phrase(global_text, p)
                       for p in legal_phrases]
})

legal_comparison

,Legal Term,Australia,Global Climate
0,Paris Agreement,2,10
1,International Court of Justice,2,3
2,ICJ,2,0
3,human rights,8,8
4,customary international law,1,1
5,advisory opinion,3,8
6,climate change,18,11
7,legal obligations,1,0
8,international law,2,4
9,Environment Protection and Biodiversity Conser...,2,0
